# 01. Failure analysis - Run All

Attach the updated code Dataset AND the original YOLO26m VisDrone640 best.pt (or checkpoint ZIP). Prepared VisDrone is optional. Enable Internet/GPU and Run All.

Only three fixed frames, full640/full1280/tile640 overlap20%. No training, no full1610-image evaluation.

## A. Setup
Verify/copy code to the writable working project. Dependencies are installed before use. The original checkpoint is checked before preparing data; if data is missing, the original CPU preparation runs automatically.

In [ ]:
import hashlib
import json
import os
import sys
import zipfile
from pathlib import Path, PurePosixPath

CODE = Path("/kaggle/working/small-object-detection-manual")
inputs = Path("/kaggle/input")
manifests = sorted(inputs.rglob("manual_code_manifest.json"))
if manifests:
    if len(manifests) != 1:
        raise RuntimeError("Attach one code Dataset; several code manifests were found.")
    source = manifests[0].parent
    manifest = json.loads(manifests[0].read_text(encoding="utf-8"))
    archive_path = None
else:
    bundles = []
    for path in sorted(inputs.rglob("*.zip")):
        with zipfile.ZipFile(path) as z:
            for name in z.namelist():
                if PurePosixPath(name).name == "manual_code_manifest.json":
                    bundles.append((path, name))
    if len(bundles) != 1:
        raise RuntimeError("Add Input: attach the updated manual_kaggle_code.zip code Dataset.")
    archive_path, manifest_name = bundles[0]
    prefix = manifest_name[: -len("manual_code_manifest.json")]
    with zipfile.ZipFile(archive_path) as z:
        if z.testzip():
            raise ValueError("Code ZIP failed its integrity check.")
        manifest = json.loads(z.read(manifest_name))
if manifest.get("notebook_mode") != "run_all_v1":
    raise RuntimeError(
        "This is the old code Dataset. Upload the updated manual_kaggle_code.zip and select its new version."
    )
files = manifest["files"]
for required in (
    "scripts/manual_experiment.py",
    "scripts/prepare_data.py",
    "notebooks/baseline.ipynb",
    "src/evaluation/metrics.py",
):
    if required not in files:
        raise ValueError(f"Incomplete code manifest: {required}")
verified = {}
for name, digest in files.items():
    relative = PurePosixPath(name)
    if relative.is_absolute() or ".." in relative.parts or "\\" in name or ":" in name:
        raise ValueError(f"Unsafe manifest path: {name}")
    if archive_path is None:
        data = (source / name).read_bytes()
    else:
        with zipfile.ZipFile(archive_path) as z:
            data = z.read(prefix + name)
    if hashlib.sha256(data).hexdigest() != digest:
        raise ValueError(f"Missing or corrupt code file: {name}")
    verified[name] = data
# Only manifest-listed code files are copied. Existing data and run results are preserved.
for name, data in verified.items():
    target = CODE / name
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != files[name]:
        target.write_bytes(data)
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
os.chdir("/kaggle/working")
print(f"CODE READY: {CODE}")

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.175", "pycocotools==2.0.11"],
    check=True,
)

In [ ]:
import subprocess
import sys
from pathlib import Path

code = Path("/kaggle/working/small-object-detection-manual")
subprocess.run(
    [sys.executable, str(code / "scripts/manual_experiment.py"), "setup", "--failure-checkpoint"],
    check=True,
)

### Optional reusable prepared-data export (Run All skips)

In [ ]:
CREATE_PREPARED_ZIP = False  # Set True only for this optional/manual action.
if CREATE_PREPARED_ZIP:
    import subprocess
    import sys
    from pathlib import Path

    code = Path("/kaggle/working/small-object-detection-manual")
    import os

    from IPython.display import FileLink, display

    result = subprocess.check_output(
        [sys.executable, str(code / "scripts/manual_experiment.py"), "prepare"], text=True
    )
    print(result)
    display(FileLink(os.path.relpath(result.strip().splitlines()[-1])))
else:
    print("Skipped: optional/manual action, or Training is not complete.")

## Three failure comparisons
9999938_00000_d_0000207.jpg; 9999986_00000_d_0000043.jpg; 0000006_00611_d_0000002.jpg.
ROI, confidence/IoU, localization TP/FP/FN, class correctness, before/after cross-tile NMS. Completed results are integrity-checked and reused.

In [ ]:
import subprocess
import sys
from pathlib import Path

code = Path("/kaggle/working/small-object-detection-manual")
subprocess.run([sys.executable, str(code / "scripts/manual_experiment.py"), "failure"], check=True)

## Final Export
Download the ZIP with figures, arrays, diagnostics and provenance.

In [ ]:
import os
import zipfile
from datetime import UTC, datetime
from pathlib import Path

from IPython.display import FileLink, display

folder = Path("/kaggle/working/manual_experiments/failure_analysis")
if not (folder / "output_manifest.json").is_file():
    raise RuntimeError("Failure comparisons did not finish; no final archive was created.")
archive = folder.parent / (
    "failure_analysis_" + datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ") + ".zip"
)
with zipfile.ZipFile(archive, "x", compression=zipfile.ZIP_DEFLATED) as z:
    for path in folder.rglob("*"):
        if path.is_file():
            z.write(path, folder.name + "/" + path.relative_to(folder).as_posix())
with zipfile.ZipFile(archive) as z:
    if z.testzip():
        raise ValueError("Failure export failed CRC validation.")
display(FileLink(os.path.relpath(archive)))

Limits of the original evaluator: class-agnostic localization recall; simplified box-IoU ignore filtering; COCO-style AP with custom cap1000.